# Exploratory Data Analysis & Uncertainty Quantification
### Course: AI Tools for Research (ATR) — Lab 02
**Researcher:** Rajhans (`PAI2026001`), Department of AI & Trustworthy Computing, IIIT Allahabad

This interactive notebook explores:
1. Dataset structure and sentence-level human annotations (`wikibio_processed.json`).
2. Token-level negative log-probability distributions across factual categories.
3. Demonstrating `token_keyword` stopword filtering and `token_hybrid` scoring.
4. Evaluating ROC / PR curves and visualizing the Cost-Quality Pareto Frontier.

In [ ]:
import os
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Set project root
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, PROJECT_ROOT)

from src.preprocessing import load_processed_dataset, audit_dataset_integrity

data_path = os.path.join(PROJECT_ROOT, "data", "processed", "wikibio_processed.json")
records = load_processed_dataset(data_path)
audit = audit_dataset_integrity(records)
print(f"Passages: {audit['num_passages']}, Total Sentences: {audit['total_sentences']}")
print(f"Annotation distribution: {audit['annotations']}")

## 1. Inspect Passage 0 and Sentence-to-Token Alignments

In [ ]:
p0 = records[0]
print(f"Passage 0 Target Concept: {p0['concept_name']}")
print(f"Generated Text:\n{p0['gpt3_text']}\n")

print("Sentence Breakdowns:")
for s_idx, s in enumerate(p0['sentences']):
    print(f"[Sent {s_idx}] ({s['annotation']}): {s['text']}")
    print(f"    Token Count: {len(s['token_logprobs'])}")
    print(f"    Sample Tokens: {s['tokens'][:5]}...")

## 2. Load Experimental Predictions and Compare Distributions

In [ ]:
preds_csv = os.path.join(PROJECT_ROOT, "results", "predictions", "experiment_predictions.csv")
df_preds = pd.read_csv(preds_csv)
print(f"Loaded {len(df_preds)} sentence predictions.")
display(df_preds[['annotation', 'token_mean', 'token_min', 'token_keyword', 'token_hybrid_a70', 'selfcheck_1gram_max_N20', 'selfcheck_tfidf_N20']].groupby('annotation').mean())

## 3. Visualize Token Uncertainty Distribution by Annotation Class

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Token Mean Distribution
for cat in ['accurate', 'minor_inaccurate', 'major_inaccurate']:
    sub = df_preds[df_preds['annotation'] == cat]
    axes[0].hist(sub['token_mean'], bins=30, alpha=0.5, label=cat, density=True)
axes[0].set_title("Distribution of Token Mean Uncertainty")
axes[0].set_xlabel("Token Mean Negative Log-Probability")
axes[0].set_ylabel("Density")
axes[0].legend()

# Proposed Token Hybrid Distribution
for cat in ['accurate', 'minor_inaccurate', 'major_inaccurate']:
    sub = df_preds[df_preds['annotation'] == cat]
    axes[1].hist(sub['token_hybrid_a70'], bins=30, alpha=0.5, label=cat, density=True)
axes[1].set_title("Distribution of Proposed Token Hybrid (a=0.70)")
axes[1].set_xlabel("Hybrid Uncertainty Score")
axes[1].legend()

plt.tight_layout()
plt.show()

## 4. Benchmark Results Summary & Pareto Frontier View

In [ ]:
results_csv = os.path.join(PROJECT_ROOT, "results", "results.csv")
df_res = pd.read_csv(results_csv)

# View NonFact Protocol Leaderboard
df_nf = df_res[df_res['protocol'] == 'nonfact'].sort_values('auroc', ascending=False)
display(df_nf[['method', 'generation_passes', 'latency_ms_per_passage', 'auroc', 'auc_pr', 'fpr95', 'prec_at_rec80']])